# 04 - The Reparameterization Trick

In the previous notebook, variational inference turned posterior approximation into an optimization problem.
The next question is: **how do we compute gradients through random sampling?**

This notebook covers:
1. Why naive sampling breaks differentiation
2. The reparameterization idea
3. A Gaussian example
4. Why this gives lower-variance gradient estimates

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Part 1: The problem

Suppose our variational distribution is Gaussian:
$$q(z) = athcal{N}(z; u, igma^2)$$

and our objective depends on an expectation under $q$:
$$
athbb{E}_{q(z)}[f(z)]
$$

If we sample directly as $z im athcal{N}(u, igma^2)$, the random draw seems to depend on $u$ and $igma$ in a way that is awkward for optimization.

The reparameterization trick rewrites the randomness so the parameters appear in a deterministic transformation.

## Part 2: The key idea

Instead of sampling $z$ directly, sample from a parameter-free noise source:
$$
psilon im athcal{N}(0, 1)
$$
and transform it as
$$
z = u + igma psilon
$$

Now the randomness is isolated in $psilon$, and $z$ is a differentiable function of $u$ and $igma$.

In [ ]:
mu = 1.5
sigma = 0.8
n = 10000

# Direct sampling
z_direct = np.random.normal(loc=mu, scale=sigma, size=n)

# Reparameterized sampling
eps = np.random.normal(loc=0.0, scale=1.0, size=n)
z_reparam = mu + sigma * eps

x = np.linspace(-2, 5, 500)
pdf = stats.norm.pdf(x, loc=mu, scale=sigma)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(z_direct, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].plot(x, pdf, color='navy', linewidth=2)
axes[0].set_title('Direct sampling')
axes[0].set_xlabel('z')

axes[1].hist(z_reparam, bins=50, density=True, alpha=0.7, color='darkorange', edgecolor='black')
axes[1].plot(x, pdf, color='navy', linewidth=2)
axes[1].set_title('Reparameterized sampling: z = mu + sigma * eps')
axes[1].set_xlabel('z')

plt.tight_layout()
plt.show()

print(f'Direct sample mean: {np.mean(z_direct):.3f}, std: {np.std(z_direct):.3f}')
print(f'Reparam sample mean: {np.mean(z_reparam):.3f}, std: {np.std(z_reparam):.3f}')

## Part 3: Using it inside variational inference

For a Gaussian variational family, the ELBO can be written as
$$
athcal{L}(q) = athbb{E}_{q(z)}[og p(x, z) - og q(z)]
$$

Using reparameterization, this becomes an expectation over standard Gaussian noise:
$$
athcal{L}(q) = athbb{E}_{psilon im athcal{N}(0,1)}[og p(x, u + igma psilon) - og q(u + igma psilon)]
$$

That makes Monte Carlo estimates compatible with gradient-based optimization.

In [ ]:
# Simple target posterior-like distribution: p(z) = N(2.0, 0.6^2)
# We will fit a Gaussian variational distribution q(z) = N(mu, sigma^2) to it.

target_mean = 2.0
target_std = 0.6

def log_p(z):
    return stats.norm.logpdf(z, loc=target_mean, scale=target_std)

def log_q(z, mu, sigma):
    return stats.norm.logpdf(z, loc=mu, scale=sigma)

fixed_eps = np.random.normal(size=4000)

def estimate_elbo_reparam(params):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    z = mu + sigma * fixed_eps
    return np.mean(log_p(z) - log_q(z, mu, sigma))

def objective(params):
    return -estimate_elbo_reparam(params)

In [ ]:
initial_params = np.array([0.0, np.log(1.5)])
result = minimize(objective, initial_params, method='Nelder-Mead', options={'maxiter': 300, 'xatol': 1e-3, 'fatol': 1e-3})

mu_opt, log_sigma_opt = result.x
sigma_opt = np.exp(log_sigma_opt)

print('Optimization success:', result.success)
print(f'Estimated mu:    {mu_opt:.3f}')
print(f'Estimated sigma: {sigma_opt:.3f}')
print(f'True mean:       {target_mean:.3f}')
print(f'True std:        {target_std:.3f}')
print(f'Estimated ELBO:  {-result.fun:.4f}')

In [ ]:
grid = np.linspace(-1, 5, 1000)
target_pdf = stats.norm.pdf(grid, loc=target_mean, scale=target_std)
q_pdf = stats.norm.pdf(grid, loc=mu_opt, scale=sigma_opt)

plt.figure(figsize=(9, 4))
plt.plot(grid, target_pdf, label='target p(z)', linewidth=3, color='navy')
plt.plot(grid, q_pdf, label='optimized q(z)', linewidth=2, color='darkorange')
plt.axvline(target_mean, color='navy', linestyle='--', alpha=0.6)
plt.axvline(mu_opt, color='darkorange', linestyle='--', alpha=0.6)
plt.xlabel('z')
plt.ylabel('density')
plt.title('Variational approximation learned via reparameterization')
plt.legend()
plt.show()

## Part 4: Why it helps with gradient variance

A major benefit is that reparameterization often gives **lower-variance gradient estimates** than score-function estimators.

Consider
$$
J(u) = athbb{E}_{z im athcal{N}(u, igma^2)}[z^2]
$$
with fixed $igma$. The exact derivative is
$$
rac{d}{du} J(u) = 2u
$$

We can compare two Monte Carlo estimators of this derivative.

In [ ]:
mu = 1.0
sigma = 1.2
true_grad = 2 * mu

def score_function_estimator(mu, sigma, n_samples):
    z = np.random.normal(mu, sigma, size=n_samples)
    f = z**2
    score = (z - mu) / (sigma**2)
    return np.mean(f * score)

def reparam_estimator(mu, sigma, n_samples):
    eps = np.random.normal(0.0, 1.0, size=n_samples)
    z = mu + sigma * eps
    grad = 2 * z  # derivative of z^2 wrt mu through z = mu + sigma * eps
    return np.mean(grad)

n_repeats = 500
n_samples = 30
score_estimates = np.array([score_function_estimator(mu, sigma, n_samples) for _ in range(n_repeats)])
reparam_estimates = np.array([reparam_estimator(mu, sigma, n_samples) for _ in range(n_repeats)])

print(f'True gradient:                 {true_grad:.3f}')
print(f'Score-function mean estimate:  {np.mean(score_estimates):.3f}')
print(f'Reparameterized mean estimate: {np.mean(reparam_estimates):.3f}')
print(f'Score-function std:            {np.std(score_estimates):.3f}')
print(f'Reparameterized std:           {np.std(reparam_estimates):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].hist(score_estimates, bins=35, color='crimson', alpha=0.7, edgecolor='black')
axes[0].axvline(true_grad, color='black', linestyle='--', linewidth=2)
axes[0].set_title('Score-function gradient estimates')
axes[0].set_xlabel('estimated gradient')

axes[1].hist(reparam_estimates, bins=35, color='seagreen', alpha=0.7, edgecolor='black')
axes[1].axvline(true_grad, color='black', linestyle='--', linewidth=2)
axes[1].set_title('Reparameterized gradient estimates')
axes[1].set_xlabel('estimated gradient')

plt.tight_layout()
plt.show()

## Summary

What to remember:
1. Reparameterization rewrites sampling as a deterministic transform of noise
2. For Gaussians, $z = u + igma psilon$ with $psilon im athcal{N}(0,1)$
3. This lets Monte Carlo objectives be optimized more effectively
4. The resulting gradient estimates often have much lower variance

This is one of the core ideas behind modern stochastic variational inference and variational autoencoders.

In [ ]:
# Exercises
# 1) Change the target_mean and target_std, then re-run the optimization.
# 2) Increase n_samples in the gradient comparison. How do both variances change?
# 3) Try a smaller sigma in the gradient experiment. What happens to estimator spread?

pass